# Train XGBoost Urgency Classifier

This notebook keeps all XGBoost urgency workflows in one place:
- a baseline non-cost-sensitive 4-class urgency model
- a cost-sensitive 4-class urgency model for imbalanced urgency labels
- a binary screening model for `routine-review` vs `immediate-attention`

The cost-sensitive section selects the best weighted multiclass model using validation metrics. The screening section tunes the decision threshold to favor urgent-class recall.


In [17]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "train":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_feature_engineered_encoded.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "train" / "artifacts"

BASELINE_ARTIFACT_DIR = ARTIFACTS_DIR / "xgboost_urgency_classifire"
BASELINE_MODEL_PATH = BASELINE_ARTIFACT_DIR / "model.pkl"
BASELINE_METADATA_PATH = BASELINE_ARTIFACT_DIR / "metadata.json"

COST_SENSITIVE_ARTIFACT_DIR = ARTIFACTS_DIR / "xgboost_urgency_classifire_cost_sensitive"
COST_SENSITIVE_MODEL_PATH = COST_SENSITIVE_ARTIFACT_DIR / "model.pkl"
COST_SENSITIVE_METADATA_PATH = COST_SENSITIVE_ARTIFACT_DIR / "metadata.json"
COST_SENSITIVE_COMPARISON_PATH = COST_SENSITIVE_ARTIFACT_DIR / "model_comparison.csv"
COST_SENSITIVE_PREDICTIONS_PATH = COST_SENSITIVE_ARTIFACT_DIR / "test_predictions.csv"

SCREENING_ARTIFACT_DIR = ARTIFACTS_DIR / "xgboost_urgency_binary_screening"
SCREENING_MODEL_PATH = SCREENING_ARTIFACT_DIR / "model.pkl"
SCREENING_METADATA_PATH = SCREENING_ARTIFACT_DIR / "metadata.json"
SCREENING_THRESHOLD_REPORT_PATH = SCREENING_ARTIFACT_DIR / "threshold_report.csv"
SCREENING_PREDICTIONS_PATH = SCREENING_ARTIFACT_DIR / "test_predictions.csv"

RANDOM_SEED = 42
TARGET_RECALL = 0.95

BASELINE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
COST_SENSITIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SCREENING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [18]:
incidents_df = pd.read_csv(DATASET_PATH)

required_columns = {"feature_concatanation", "urgency", "urgency_label"}
missing_columns = required_columns - set(incidents_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

model_df = incidents_df[["feature_concatanation", "urgency", "urgency_label"]].dropna().copy()
model_df["feature_concatanation"] = model_df["feature_concatanation"].astype(str).str.strip()
model_df = model_df[model_df["feature_concatanation"] != ""]

print(f"Loaded {len(model_df)} training rows from {DATASET_PATH}")
model_df.head()


Loaded 3600 training rows from /home/lakshan/ssp/IMS/data/processed/incidents_feature_engineered_encoded.csv


,feature_concatanation,urgency,urgency_label
0,Hospital Administrator Consultant session reve...,Low,0
1,Staff Nurse - Paediatric Ward Diet slip notes ...,High,2
2,OPD Nursing Officer Large crowd surged at OPD ...,Medium,1
3,ICU Charge Nurse ICU AHU-2 not holding 22C; te...,High,2
4,"Nursing Officer-in-Charge, Surgical Ward Publi...",High,2


In [19]:
X = model_df["feature_concatanation"]
y = model_df["urgency_label"].astype(int)

label_mapping_df = (
    model_df[["urgency", "urgency_label"]]
    .drop_duplicates()
    .sort_values("urgency_label")
    .reset_index(drop=True)
)
label_mapping = dict(zip(label_mapping_df["urgency_label"], label_mapping_df["urgency"]))

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_validation), len(X_test)],
        "ratio": [len(X_train) / len(X), len(X_validation) / len(X), len(X_test) / len(X)],
    }
)


,split,rows,ratio
0,train,2520,0.70
1,validation,540,0.15
2,test,540,0.15


## Baseline Non-Cost-Sensitive Model


In [20]:
baseline_urgency_classifier = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            XGBClassifier(
                objective="multi:softprob",
                num_class=len(label_mapping),
                n_estimators=250,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="mlogloss",
                random_state=RANDOM_SEED,
                tree_method="hist",
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_urgency_classifier.fit(X_train, y_train)
print("Finished training baseline XGBoost urgency classifier")


Finished training baseline XGBoost urgency classifier


In [21]:
baseline_validation_predictions = baseline_urgency_classifier.predict(X_validation)
baseline_test_predictions = baseline_urgency_classifier.predict(X_test)

baseline_validation_accuracy = accuracy_score(y_validation, baseline_validation_predictions)
baseline_test_accuracy = accuracy_score(y_test, baseline_test_predictions)

print(f"Validation accuracy: {baseline_validation_accuracy:.4f}")
print(f"Test accuracy: {baseline_test_accuracy:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        y_test,
        baseline_test_predictions,
        labels=sorted(label_mapping),
        target_names=[label_mapping[label] for label in sorted(label_mapping)],
        zero_division=0,
    )
)

baseline_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, baseline_test_predictions, labels=sorted(label_mapping)),
    index=[f"actual_{label_mapping[label]}" for label in sorted(label_mapping)],
    columns=[f"pred_{label_mapping[label]}" for label in sorted(label_mapping)],
)
baseline_confusion_matrix_df


Validation accuracy: 0.5741
Test accuracy: 0.5796

Test classification report:

              precision    recall  f1-score   support

         Low       0.25      0.09      0.13        11
      Medium       0.56      0.55      0.56       175
        High       0.60      0.70      0.65       270
    Critical       0.52      0.31      0.39        84

    accuracy                           0.58       540
   macro avg       0.48      0.41      0.43       540
weighted avg       0.57      0.58      0.57       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,1,5,5,0
actual_Medium,3,97,72,3
actual_High,0,60,189,21
actual_Critical,0,10,48,26


In [22]:
with BASELINE_MODEL_PATH.open("wb") as model_file:
    pickle.dump(baseline_urgency_classifier, model_file)

baseline_metadata = {
    "dataset_path": str(DATASET_PATH),
    "model_artifact_dir": str(BASELINE_ARTIFACT_DIR),
    "model_path": str(BASELINE_MODEL_PATH),
    "label_mapping": {str(key): value for key, value in label_mapping.items()},
    "random_seed": RANDOM_SEED,
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "validation_accuracy": float(baseline_validation_accuracy),
    "test_accuracy": float(baseline_test_accuracy),
    "feature_column": "feature_concatanation",
    "target_column": "urgency_label",
    "target_name_column": "urgency",
}

BASELINE_METADATA_PATH.write_text(json.dumps(baseline_metadata, indent=2))

print(f"Saved trained model to {BASELINE_MODEL_PATH}")
print(f"Saved metadata to {BASELINE_METADATA_PATH}")


Saved trained model to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire/model.pkl
Saved metadata to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire/metadata.json


## Cost-Sensitive Model Selection

This variant uses class-weighted samples so minority and higher-severity urgency classes carry more training cost. Multiple hyperparameter settings are trained, then the best model is selected using validation balanced accuracy and validation macro F1.


In [23]:
classes = np.array(sorted(label_mapping), dtype=int)
base_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_map = {label: float(weight) for label, weight in zip(classes, base_class_weights)}

# Increase the penalty for missing the highest-severity incidents.
severity_cost_multiplier = {
    label: 1.0 + (label / max(classes))
    for label in classes
}
weighted_class_map = {
    label: class_weight_map[label] * severity_cost_multiplier[label]
    for label in classes
}
sample_weights = y_train.map(weighted_class_map)

pd.DataFrame(
    {
        "urgency_label": classes,
        "urgency": [label_mapping[label] for label in classes],
        "class_weight": [class_weight_map[label] for label in classes],
        "severity_multiplier": [severity_cost_multiplier[label] for label in classes],
        "effective_weight": [weighted_class_map[label] for label in classes],
    }
)


,urgency_label,urgency,class_weight,severity_multiplier,effective_weight
0,0,Low,12.600000,1.000000,12.600000
1,1,Medium,0.773006,1.333333,1.030675
2,2,High,0.500000,1.666667,0.833333
3,3,Critical,1.594937,2.000000,3.189873


In [24]:
cost_sensitive_candidates = [
    {
        "model_name": "weighted_xgb_v1",
        "n_estimators": 350,
        "max_depth": 6,
        "learning_rate": 0.08,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 2,
    },
    {
        "model_name": "weighted_xgb_v2",
        "n_estimators": 500,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 1,
    },
]

cost_sensitive_results = []
cost_sensitive_models = {}

for candidate in cost_sensitive_candidates:
    candidate_pipeline = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=20000,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                XGBClassifier(
                    objective="multi:softprob",
                    num_class=len(label_mapping),
                    n_estimators=candidate["n_estimators"],
                    max_depth=candidate["max_depth"],
                    learning_rate=candidate["learning_rate"],
                    subsample=candidate["subsample"],
                    colsample_bytree=candidate["colsample_bytree"],
                    min_child_weight=candidate["min_child_weight"],
                    eval_metric="mlogloss",
                    random_state=RANDOM_SEED,
                    tree_method="hist",
                    n_jobs=-1,
                ),
            ),
        ]
    )

    candidate_pipeline.fit(
        X_train,
        y_train,
        classifier__sample_weight=sample_weights.to_numpy(),
    )

    validation_predictions = candidate_pipeline.predict(X_validation)
    validation_accuracy = accuracy_score(y_validation, validation_predictions)
    validation_balanced_accuracy = balanced_accuracy_score(y_validation, validation_predictions)
    validation_macro_f1 = f1_score(y_validation, validation_predictions, average="macro")

    cost_sensitive_models[candidate["model_name"]] = candidate_pipeline
    cost_sensitive_results.append(
        {
            **candidate,
            "validation_accuracy": float(validation_accuracy),
            "validation_balanced_accuracy": float(validation_balanced_accuracy),
            "validation_macro_f1": float(validation_macro_f1),
        }
    )
    print(
        candidate["model_name"],
        f"validation_balanced_accuracy={validation_balanced_accuracy:.4f}",
        f"validation_macro_f1={validation_macro_f1:.4f}",
    )

cost_sensitive_results_df = pd.DataFrame(cost_sensitive_results).sort_values(
    by=["validation_balanced_accuracy", "validation_macro_f1", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)

best_cost_sensitive_config = cost_sensitive_results_df.iloc[0].to_dict()
best_cost_sensitive_model_name = best_cost_sensitive_config["model_name"]
cost_sensitive_urgency_classifier = cost_sensitive_models[best_cost_sensitive_model_name]

print(f"Selected best cost-sensitive model: {best_cost_sensitive_model_name}")
cost_sensitive_results_df


weighted_xgb_v1 validation_balanced_accuracy=0.4429 validation_macro_f1=0.4342
weighted_xgb_v2 validation_balanced_accuracy=0.4165 validation_macro_f1=0.3992
Selected best cost-sensitive model: weighted_xgb_v1


,model_name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,validation_accuracy,validation_balanced_accuracy,validation_macro_f1
0,weighted_xgb_v1,350,6,0.08,0.90,0.90,2,0.544444,0.442891,0.434206
1,weighted_xgb_v2,500,5,0.05,0.85,0.85,1,0.533333,0.416528,0.399169


In [25]:
cost_validation_predictions = cost_sensitive_urgency_classifier.predict(X_validation)
cost_test_predictions = cost_sensitive_urgency_classifier.predict(X_test)

cost_validation_accuracy = accuracy_score(y_validation, cost_validation_predictions)
cost_test_accuracy = accuracy_score(y_test, cost_test_predictions)
cost_validation_balanced_accuracy = balanced_accuracy_score(y_validation, cost_validation_predictions)
cost_test_balanced_accuracy = balanced_accuracy_score(y_test, cost_test_predictions)
cost_validation_macro_f1 = f1_score(y_validation, cost_validation_predictions, average="macro")
cost_test_macro_f1 = f1_score(y_test, cost_test_predictions, average="macro")

cost_report_dict = classification_report(
    y_test,
    cost_test_predictions,
    labels=sorted(label_mapping),
    target_names=[label_mapping[label] for label in sorted(label_mapping)],
    zero_division=0,
    output_dict=True,
)

print(f"Selected model: {best_cost_sensitive_model_name}")
print(f"Validation accuracy: {cost_validation_accuracy:.4f}")
print(f"Test accuracy: {cost_test_accuracy:.4f}")
print(f"Validation balanced accuracy: {cost_validation_balanced_accuracy:.4f}")
print(f"Test balanced accuracy: {cost_test_balanced_accuracy:.4f}")
print(f"Validation macro F1: {cost_validation_macro_f1:.4f}")
print(f"Test macro F1: {cost_test_macro_f1:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        y_test,
        cost_test_predictions,
        labels=sorted(label_mapping),
        target_names=[label_mapping[label] for label in sorted(label_mapping)],
        zero_division=0,
    )
)

cost_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, cost_test_predictions, labels=sorted(label_mapping)),
    index=[f"actual_{label_mapping[label]}" for label in sorted(label_mapping)],
    columns=[f"pred_{label_mapping[label]}" for label in sorted(label_mapping)],
)
cost_confusion_matrix_df


Selected model: weighted_xgb_v1
Validation accuracy: 0.5444
Test accuracy: 0.5741
Validation balanced accuracy: 0.4429
Test balanced accuracy: 0.4441
Validation macro F1: 0.4342
Test macro F1: 0.4501

Test classification report:

              precision    recall  f1-score   support

         Low       0.25      0.09      0.13        11
      Medium       0.55      0.63      0.59       175
        High       0.63      0.59      0.61       270
    Critical       0.48      0.46      0.47        84

    accuracy                           0.57       540
   macro avg       0.48      0.44      0.45       540
weighted avg       0.57      0.57      0.57       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,1,5,5,0
actual_Medium,3,110,57,5
actual_High,0,72,160,38
actual_Critical,0,13,32,39


In [26]:
with COST_SENSITIVE_MODEL_PATH.open("wb") as model_file:
    pickle.dump(cost_sensitive_urgency_classifier, model_file)

cost_predictions_df = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual_label": y_test.reset_index(drop=True),
        "predicted_label": pd.Series(cost_test_predictions),
    }
)
cost_predictions_df["actual_urgency"] = cost_predictions_df["actual_label"].map(label_mapping)
cost_predictions_df["predicted_urgency"] = cost_predictions_df["predicted_label"].map(label_mapping)

cost_sensitive_metadata = {
    "dataset_path": str(DATASET_PATH),
    "model_artifact_dir": str(COST_SENSITIVE_ARTIFACT_DIR),
    "model_path": str(COST_SENSITIVE_MODEL_PATH),
    "comparison_path": str(COST_SENSITIVE_COMPARISON_PATH),
    "predictions_path": str(COST_SENSITIVE_PREDICTIONS_PATH),
    "selected_model_name": best_cost_sensitive_model_name,
    "selected_model_config": best_cost_sensitive_config,
    "candidate_results": cost_sensitive_results,
    "label_mapping": {str(key): value for key, value in label_mapping.items()},
    "class_weights": {str(int(key)): value for key, value in weighted_class_map.items()},
    "random_seed": RANDOM_SEED,
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "validation_accuracy": float(cost_validation_accuracy),
    "test_accuracy": float(cost_test_accuracy),
    "validation_balanced_accuracy": float(cost_validation_balanced_accuracy),
    "test_balanced_accuracy": float(cost_test_balanced_accuracy),
    "validation_macro_f1": float(cost_validation_macro_f1),
    "test_macro_f1": float(cost_test_macro_f1),
    "feature_column": "feature_concatanation",
    "target_column": "urgency_label",
    "target_name_column": "urgency",
    "classification_report": cost_report_dict,
    "confusion_matrix": cost_confusion_matrix_df.to_dict(),
}

COST_SENSITIVE_METADATA_PATH.write_text(json.dumps(cost_sensitive_metadata, indent=2))
cost_sensitive_results_df.to_csv(COST_SENSITIVE_COMPARISON_PATH, index=False)
cost_predictions_df.to_csv(COST_SENSITIVE_PREDICTIONS_PATH, index=False)

print(f"Saved trained model to {COST_SENSITIVE_MODEL_PATH}")
print(f"Saved metadata to {COST_SENSITIVE_METADATA_PATH}")
print(f"Saved model comparison to {COST_SENSITIVE_COMPARISON_PATH}")
print(f"Saved test predictions to {COST_SENSITIVE_PREDICTIONS_PATH}")


Saved trained model to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire_cost_sensitive/model.pkl
Saved metadata to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire_cost_sensitive/metadata.json
Saved model comparison to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire_cost_sensitive/model_comparison.csv
Saved test predictions to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_classifire_cost_sensitive/test_predictions.csv


## Binary Screening Model

Collapsing the original 4-tier urgency scheme into two classes makes the task much easier for the model:
- `Low` and `Medium` become one routine bucket
- `High` and `Critical` become one urgent bucket

This turns the model into a screening filter. The main risk becomes missing an actually urgent incident, so threshold selection emphasizes urgent-class recall rather than plain accuracy.


In [ ]:
screening_df = model_df.copy()
positive_classes = {"High", "Critical"}
screening_df["screening_label"] = screening_df["urgency"].isin(positive_classes).astype(int)
screening_df["screening_target"] = screening_df["screening_label"].map({0: "routine-review", 1: "immediate-attention"})

X_screen = screening_df["feature_concatanation"]
y_screen = screening_df["screening_label"].astype(int)
screening_label_mapping = {0: "routine-review", 1: "immediate-attention"}

X_train_screen, X_temp_screen, y_train_screen, y_temp_screen = train_test_split(
    X_screen,
    y_screen,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_screen,
)

X_validation_screen, X_test_screen, y_validation_screen, y_test_screen = train_test_split(
    X_temp_screen,
    y_temp_screen,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp_screen,
)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train_screen), len(X_validation_screen), len(X_test_screen)],
        "positive_rows": [int(y_train_screen.sum()), int(y_validation_screen.sum()), int(y_test_screen.sum())],
        "negative_rows": [
            int((1 - y_train_screen).sum()),
            int((1 - y_validation_screen).sum()),
            int((1 - y_test_screen).sum()),
        ],
    }
)


In [ ]:
distribution_df = pd.DataFrame(
    {
        "count": screening_df["screening_label"].value_counts().sort_index(),
    }
)
distribution_df.index = distribution_df.index.map(screening_label_mapping)
distribution_df["ratio"] = distribution_df["count"] / distribution_df["count"].sum()
distribution_df


### Train Binary Screening Model

The model predicts the probability that an incident belongs to the urgent `immediate-attention` bucket. Threshold selection happens after training.


In [ ]:
screening_classifier = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                n_estimators=350,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.9,
                colsample_bytree=0.9,
                min_child_weight=2,
                reg_alpha=0.0,
                reg_lambda=1.0,
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                tree_method="hist",
                n_jobs=-1,
            ),
        ),
    ]
)

screening_classifier.fit(X_train_screen, y_train_screen)
print("Finished training binary XGBoost screening classifier")


### Threshold Selection

The notebook scans a threshold range and selects the threshold that:
- first satisfies the urgent-class recall target of at least `0.95`
- then maximizes precision
- then uses F1 as a tie-breaker

If no threshold reaches the recall target, it falls back to the threshold with the highest recall and then highest precision.


In [ ]:
validation_probabilities = screening_classifier.predict_proba(X_validation_screen)[:, 1]
test_probabilities = screening_classifier.predict_proba(X_test_screen)[:, 1]

threshold_candidates = np.round(np.arange(0.20, 0.61, 0.01), 2)
threshold_rows = []

for threshold in threshold_candidates:
    validation_predictions = (validation_probabilities >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "validation_precision": float(precision_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_recall": float(recall_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_f1": float(f1_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_accuracy": float(accuracy_score(y_validation_screen, validation_predictions)),
        }
    )

threshold_report_df = pd.DataFrame(threshold_rows)

eligible_thresholds = threshold_report_df[threshold_report_df["validation_recall"] >= TARGET_RECALL]
if not eligible_thresholds.empty:
    selected_threshold_row = eligible_thresholds.sort_values(
        by=["validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, True],
    ).iloc[0]
else:
    selected_threshold_row = threshold_report_df.sort_values(
        by=["validation_recall", "validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, False, True],
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])
validation_pr_auc = average_precision_score(y_validation_screen, validation_probabilities)
test_pr_auc = average_precision_score(y_test_screen, test_probabilities)

print(f"Selected threshold: {selected_threshold:.2f}")
print(f"Validation PR-AUC: {validation_pr_auc:.4f}")
print(f"Test PR-AUC: {test_pr_auc:.4f}")

threshold_report_df


In [ ]:
default_validation_predictions = (validation_probabilities >= 0.50).astype(int)
selected_validation_predictions = (validation_probabilities >= selected_threshold).astype(int)
selected_test_predictions = (test_probabilities >= selected_threshold).astype(int)

default_validation_recall = recall_score(y_validation_screen, default_validation_predictions, zero_division=0)
selected_validation_recall = recall_score(y_validation_screen, selected_validation_predictions, zero_division=0)

test_accuracy = accuracy_score(y_test_screen, selected_test_predictions)
test_precision = precision_score(y_test_screen, selected_test_predictions, zero_division=0)
test_recall = recall_score(y_test_screen, selected_test_predictions, zero_division=0)
test_f1 = f1_score(y_test_screen, selected_test_predictions, zero_division=0)

test_report_dict = classification_report(
    y_test_screen,
    selected_test_predictions,
    labels=[0, 1],
    target_names=[screening_label_mapping[0], screening_label_mapping[1]],
    zero_division=0,
    output_dict=True,
)

print(f"Default threshold recall (0.50): {default_validation_recall:.4f}")
print(f"Selected threshold recall ({selected_threshold:.2f}): {selected_validation_recall:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision for immediate-attention: {test_precision:.4f}")
print(f"Test recall for immediate-attention: {test_recall:.4f}")
print(f"Test F1 for immediate-attention: {test_f1:.4f}")
print(f"Test PR-AUC: {test_pr_auc:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        y_test_screen,
        selected_test_predictions,
        labels=[0, 1],
        target_names=[screening_label_mapping[0], screening_label_mapping[1]],
        zero_division=0,
    )
)

test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test_screen, selected_test_predictions, labels=[0, 1]),
    index=[f"actual_{screening_label_mapping[0]}", f"actual_{screening_label_mapping[1]}"],
    columns=[f"pred_{screening_label_mapping[0]}", f"pred_{screening_label_mapping[1]}"],
)
test_confusion_matrix_df


## Workflow Fit

This binary model fits best when the urgent queue still receives human review. In that setup, the model acts as an automated screening layer:
- routine-looking incidents stay in the `routine-review` bucket
- anything with meaningful urgency probability is escalated into the `immediate-attention` bucket

That keeps the final fine-grained triage decision with staff while reducing the chance that a truly urgent incident waits in a routine queue.


In [ ]:
with SCREENING_MODEL_PATH.open("wb") as model_file:
    pickle.dump(screening_classifier, model_file)

screening_predictions_df = pd.DataFrame(
    {
        "text": X_test_screen.reset_index(drop=True),
        "actual_label": y_test_screen.reset_index(drop=True),
        "predicted_probability_immediate_attention": pd.Series(test_probabilities),
        "predicted_label": pd.Series(selected_test_predictions),
    }
)
screening_predictions_df["actual_target"] = screening_predictions_df["actual_label"].map(screening_label_mapping)
screening_predictions_df["predicted_target"] = screening_predictions_df["predicted_label"].map(screening_label_mapping)

metadata = {
    "dataset_path": str(DATASET_PATH),
    "model_artifact_dir": str(SCREENING_ARTIFACT_DIR),
    "model_path": str(SCREENING_MODEL_PATH),
    "threshold_report_path": str(SCREENING_THRESHOLD_REPORT_PATH),
    "predictions_path": str(SCREENING_PREDICTIONS_PATH),
    "random_seed": RANDOM_SEED,
    "target_recall": TARGET_RECALL,
    "selected_threshold": selected_threshold,
    "positive_class": screening_label_mapping[1],
    "negative_class": screening_label_mapping[0],
    "train_rows": int(len(X_train_screen)),
    "validation_rows": int(len(X_validation_screen)),
    "test_rows": int(len(X_test_screen)),
    "validation_pr_auc": float(validation_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "test_accuracy": float(test_accuracy),
    "test_precision_immediate_attention": float(test_precision),
    "test_recall_immediate_attention": float(test_recall),
    "test_f1_immediate_attention": float(test_f1),
    "feature_column": "feature_concatanation",
    "target_column": "screening_label",
    "target_name_column": "screening_target",
    "classification_report": test_report_dict,
    "confusion_matrix": test_confusion_matrix_df.to_dict(),
}

SCREENING_METADATA_PATH.write_text(json.dumps(metadata, indent=2))
threshold_report_df.to_csv(SCREENING_THRESHOLD_REPORT_PATH, index=False)
screening_predictions_df.to_csv(SCREENING_PREDICTIONS_PATH, index=False)

print(f"Saved trained model to {SCREENING_MODEL_PATH}")
print(f"Saved metadata to {SCREENING_METADATA_PATH}")
print(f"Saved threshold report to {SCREENING_THRESHOLD_REPORT_PATH}")
print(f"Saved test predictions to {SCREENING_PREDICTIONS_PATH}")
